# Environmental Impact Analysis on Kaggle with Gemma 4

This is a Kaggle-ready, self-contained version of the notebook. It keeps the latest Gemma 4 12B model as the first choice and automatically falls back to Gemma 4 E4B if the runtime is too small.

## Kaggle notes
- Enable GPU in Kaggle before running this notebook.
- `GEMMA_MODEL_ID` can be set to force a specific checkpoint.
- The reference smokestacks image is embedded in the notebook, so no extra asset upload is required.
- Structured JSON is saved to `/kaggle/working/gemma4_eia_analysis.json` when available.

In [ ]:
%pip install -q --upgrade torch torchvision "transformers>=5.5.0" accelerate sentencepiece protobuf pillow

In [ ]:
from __future__ import annotations

import base64
import json
import os
import re
import textwrap
from io import BytesIO
from pathlib import Path
from typing import Any

import torch
from IPython.display import display
from PIL import Image
from transformers import AutoModelForMultimodalLM, AutoProcessor

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_PATH = WORKDIR / "gemma4_eia_analysis.json"
MAX_NEW_TOKENS = 300

MODEL_CANDIDATES = list(dict.fromkeys([
    os.environ.get("GEMMA_MODEL_ID", "google/gemma-4-12B-it"),
    "google/gemma-4-E4B-it",
]))

EMBEDDED_IMAGE_B64 = """
...
RRRBEFFEH//Z
"""

def load_reference_image() -> Image.Image:
    raw = base64.b64decode("".join(EMBEDDED_IMAGE_B64.split()))
    return Image.open(BytesIO(raw)).convert("RGB")

def extract_json_block(text: str) -> dict[str, Any] | None:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(cleaned[start : end + 1])
    except json.JSONDecodeError:
        return None

def build_chat_text(prompt: str, include_image: bool) -> str:
    content = []
    if include_image:
        content.append({"type": "image"})
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def generate(prompt: str, image: Image.Image | None = None, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    chat_text = build_chat_text(prompt, include_image=image is not None)
    if image is None:
        inputs = processor(text=chat_text, return_tensors="pt").to(model.device)
    else:
        inputs = processor(images=image, text=chat_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

def load_model(candidates: list[str]):
    errors: list[str] = []
    for candidate in candidates:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            proc = AutoProcessor.from_pretrained(candidate, padding_side="left")
            mdl = AutoModelForMultimodalLM.from_pretrained(
                candidate,
                dtype="auto",
                device_map="auto",
            )
            mdl.eval()
            return candidate, proc, mdl
        except Exception as exc:
            errors.append(f"{candidate}: {exc}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError("Could not load any Gemma candidate:\n" + "\n".join(errors))

if not torch.cuda.is_available():
    print("Warning: Kaggle GPU is not enabled. This notebook is intended for GPU execution.")

In [ ]:
MODEL_ID, processor, model = load_model(MODEL_CANDIDATES)
print(f"Loaded model: {MODEL_ID}")

In [ ]:
reference_image = load_reference_image()
display(reference_image)
print(f"Reference image loaded in memory ({reference_image.size[0]}x{reference_image.size[1]}).")

In [ ]:
smoke_test = generate(
    "In two sentences, explain why multimodal evidence is useful in environmental impact assessment.",
    max_new_tokens=80,
)
print(smoke_test)

In [ ]:
EIA_PROMPT = textwrap.dedent("""
You are a senior environmental impact analyst.

Inspect the image and return ONLY valid JSON with these keys:
- hazard_level: one of ["low", "medium", "high", "critical"]
- summary: one sentence
- visible_evidence: 3 to 5 short strings
- likely_impact_factors: 3 to 5 short strings
- likely_processes: 2 to 4 short strings
- recommendations: 3 to 5 short strings
- uncertainty: 1 to 3 short strings
- confidence: a number from 0 to 1

Rules:
- Do not use markdown or code fences.
- Do not invent details that are not clearly supported by the image.
- If the image is ambiguous, say so in uncertainty.
""").strip()

raw_analysis = generate(EIA_PROMPT, image=reference_image, max_new_tokens=MAX_NEW_TOKENS)
print(raw_analysis)

In [ ]:
analysis = extract_json_block(raw_analysis)

if analysis is None:
    print("Could not parse JSON. Inspect raw_analysis above.")
else:
    print(json.dumps(analysis, indent=2, ensure_ascii=False))
    OUTPUT_PATH.write_text(json.dumps(analysis, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved structured result to {OUTPUT_PATH}")

## Optional tweaks
- To force a specific model, set `GEMMA_MODEL_ID` before loading.
- If Kaggle runs out of memory on 12B, the notebook will fall back to `google/gemma-4-E4B-it`.
- Increase `MAX_NEW_TOKENS` for longer analyses.